In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

PROJECT_ROOT = Path(".")

print("Python executable:")
print(sys.executable)

print("\nProject exists:", PROJECT_ROOT.exists())
print("Current project:", PROJECT_ROOT)

۲. چک خروجی positiveها

In [ ]:
files = [
    PROJECT_ROOT / "Data_proc/positives/positive_e3.csv",
    PROJECT_ROOT / "Data_proc/positives/positive_dub.csv",
    PROJECT_ROOT / "Data_proc/positives/positive_all.csv",
]

for f in files:
    df = pd.read_csv(f)
    print("\n" + "="*80)
    print("FILE:", f.name)
    print("shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nHead:")
    display(df.head())
    print("\nClass counts:")
    print(df["enzyme_class"].value_counts(dropna=False))
    print("\nUnique pair_id:", df["pair_id"].nunique())
    print("Duplicate pair_id rows:", df.duplicated("pair_id").sum())

۳. پیدا کردن duplicateهای positive_all

In [ ]:
positive_all = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_all.csv")

dup_all = positive_all[
    positive_all.duplicated("pair_id", keep=False)
].sort_values("pair_id")

print("Number of duplicated rows in positive_all:", len(dup_all))
display(dup_all)

dup_all.to_csv(
    PROJECT_ROOT / "Data_proc/qc_reports/positive_all_duplicate_pair_id_rows.csv",
    index=False
)

۴. چک اینکه آیا DUB اشتباهی هنوز E3 شده یا نه

In [ ]:
e3 = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3.csv")
dub = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub.csv")

print("E3 enzyme_class values:")
print(e3["enzyme_class"].value_counts(dropna=False))

print("\nDUB enzyme_class values:")
print(dub["enzyme_class"].value_counts(dropna=False))

print("\nFirst DUB rows:")
display(dub.head(10))

print("\nDUB genes sample:")
display(dub[["enzyme_class", "enz_ac", "enz_gene", "sub_ac", "sub_gene"]].sample(10, random_state=42))

۵. چک ستون‌های فایل خام داخل Jupyter

In [ ]:
raw_files = [
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/E3-substrate-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/DUB-substrate-interactions.txt",
]

for f in raw_files:
    print("\n" + "="*80)
    print("FILE:", f)
    
    df = pd.read_csv(f, sep=None, engine="python", dtype=str, nrows=5)
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nHead:")
    display(df)

قدم ۱

این سلول را داخل Jupyter اجرا کن

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

e3 = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3.csv")
dub = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub.csv")
all_pos = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_all.csv")

for name, df in [("E3", e3), ("DUB", dub), ("ALL", all_pos)]:
    print("\n" + "="*80)
    print(name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("enzyme_class counts:")
    print(df["enzyme_class"].value_counts(dropna=False))
    print("missing enzyme_class:", df["enzyme_class"].isna().sum())
    print("pair_id starts with nan:", df["pair_id"].astype(str).str.startswith("nan|").sum())
    display(df.head())

قدم ۲

جفت‌های مشکل‌دار را در فایل‌های E3 و DUB پیدا کن

In [ ]:
problem_pairs = [
    ("P21580", "Q13546"),
    ("Q96F44", "Q16828"),
]

for enz_ac, sub_ac in problem_pairs:
    print("\n" + "="*80)
    print("PAIR:", enz_ac, sub_ac)

    print("\nE3:")
    display(e3[(e3["enz_ac"] == enz_ac) & (e3["sub_ac"] == sub_ac)])

    print("\nDUB:")
    display(dub[(dub["enz_ac"] == enz_ac) & (dub["sub_ac"] == sub_ac)])

    print("\nALL:")
    display(all_pos[(all_pos["enz_ac"] == enz_ac) & (all_pos["sub_ac"] == sub_ac)])

قدم ۳

فعلاً یک repair سریع و امن انجام بده

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

e3_path = PROJECT_ROOT / "Data_proc/positives/positive_e3.csv"
dub_path = PROJECT_ROOT / "Data_proc/positives/positive_dub.csv"

e3 = pd.read_csv(e3_path)
dub = pd.read_csv(dub_path)

# Force correct enzyme_class
e3["enzyme_class"] = "E3"
dub["enzyme_class"] = "DUB"

# Rebuild stable IDs
for df in [e3, dub]:
    df["enz_ac"] = df["enz_ac"].astype(str).str.strip()
    df["sub_ac"] = df["sub_ac"].astype(str).str.strip()

    df["pair_id"] = (
        df["enzyme_class"].astype(str)
        + "|"
        + df["enz_ac"].astype(str)
        + "|"
        + df["sub_ac"].astype(str)
    )

    df["group_id"] = (
        df["enzyme_class"].astype(str)
        + "|"
        + df["enz_ac"].astype(str)
    )

# Column order
cols = [
    "pair_id",
    "group_id",
    "enzyme_class",
    "enz_ac",
    "sub_ac",
    "enz_gene",
    "sub_gene",
    "enzyme_type",
    "label",
    "source",
    "pmid",
]

e3 = e3[cols]
dub = dub[cols]

positive_all_fixed = pd.concat([e3, dub], ignore_index=True)

# Check exact duplicate pair_id
dup_fixed = positive_all_fixed[
    positive_all_fixed.duplicated("pair_id", keep=False)
].sort_values("pair_id")

print("Fixed positive_all shape:", positive_all_fixed.shape)
print("Unique pair_id:", positive_all_fixed["pair_id"].nunique())
print("Duplicate pair_id rows:", len(dup_fixed))
print("Missing enzyme_class:", positive_all_fixed["enzyme_class"].isna().sum())
print("pair_id starts with nan:", positive_all_fixed["pair_id"].astype(str).str.startswith("nan|").sum())

display(dup_fixed)

# Save backup of old file
old_path = PROJECT_ROOT / "Data_proc/positives/positive_all.csv"
backup_path = PROJECT_ROOT / "Data_proc/positives/positive_all_before_fix.csv"

old = pd.read_csv(old_path)
old.to_csv(backup_path, index=False)

# Save fixed file
positive_all_fixed.to_csv(old_path, index=False)

# Save QC duplicate file
dup_fixed.to_csv(
    PROJECT_ROOT / "Data_proc/qc_reports/positive_all_duplicate_pair_id_rows_after_fix.csv",
    index=False
)

print("\nSaved fixed positive_all.csv")
print("Backup saved as positive_all_before_fix.csv")

قدم ۴ 

QC را دوباره بساز

In [ ]:
def qc_row(df, name):
    return {
        "dataset": name,
        "n_rows": len(df),
        "n_unique_pair_id": df["pair_id"].nunique(),
        "n_duplicate_pair_id_rows": int(df.duplicated("pair_id").sum()),
        "n_unique_group_id": df["group_id"].nunique(),
        "n_unique_enz_ac": df["enz_ac"].nunique(),
        "n_unique_sub_ac": df["sub_ac"].nunique(),
        "n_missing_enzyme_class": int(df["enzyme_class"].isna().sum()),
        "n_missing_enz_ac": int(df["enz_ac"].isna().sum()),
        "n_missing_sub_ac": int(df["sub_ac"].isna().sum()),
        "n_missing_enz_gene": int(df["enz_gene"].isna().sum()),
        "n_missing_sub_gene": int(df["sub_gene"].isna().sum()),
        "n_missing_enzyme_type": int(df["enzyme_type"].isna().sum()),
        "n_missing_pmid": int(df["pmid"].isna().sum()),
    }

e3 = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3.csv")
dub = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub.csv")
all_fixed = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_all.csv")

qc = pd.DataFrame([
    qc_row(e3, "E3_positive"),
    qc_row(dub, "DUB_positive"),
    qc_row(all_fixed, "ALL_positive_fixed"),
])

display(qc)

qc.to_csv(PROJECT_ROOT / "Data_proc/qc_reports/positive_qc_fixed.csv", index=False)

قدم ۵

raw columnها را داخل Jupyter چک کن

In [ ]:
raw_files = [
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/E3-substrate-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/DUB-substrate-interactions.txt",
]

for f in raw_files:
    print("\n" + "="*100)
    print("FILE:", f.name)
    
    df = pd.read_csv(f, sep=None, engine="python", dtype=str, nrows=5)
    
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    
    display(df)

چک نهایی positiveها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

e3 = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3.csv")
dub = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub.csv")
pos = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_all.csv")

for name, df in [("E3", e3), ("DUB", dub), ("ALL", pos)]:
    print("\n" + "="*80)
    print(name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("\nenzyme_class counts:")
    print(df["enzyme_class"].value_counts(dropna=False))
    print("\nlabel counts:")
    print(df["label"].value_counts(dropna=False))
    print("\nunique pair_id:", df["pair_id"].nunique())
    print("duplicate pair_id rows:", df.duplicated("pair_id").sum())
    print("missing enzyme_class:", df["enzyme_class"].isna().sum())
    print("missing enz_ac:", df["enz_ac"].isna().sum())
    print("missing sub_ac:", df["sub_ac"].isna().sum())
    print("pair_id starts with nan:", df["pair_id"].astype(str).str.startswith("nan|").sum())
    display(df.head(5))

چک جفت‌های مشکل‌دار قبلی

In [ ]:
problem_pairs = [
    ("P21580", "Q13546"),
    ("Q96F44", "Q16828"),
]

for enz_ac, sub_ac in problem_pairs:
    print("\n" + "="*80)
    print("PAIR:", enz_ac, "->", sub_ac)

    tmp = pos[(pos["enz_ac"] == enz_ac) & (pos["sub_ac"] == sub_ac)]
    display(tmp[[
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "enzyme_type",
        "label",
        "pmid"
    ]])

چک تداخل E3 و DUB از نظر accession

In [ ]:
e3_enz = set(e3["enz_ac"].dropna().astype(str))
dub_enz = set(dub["enz_ac"].dropna().astype(str))

overlap_enz = sorted(e3_enz & dub_enz)

print("Number of enzyme accessions appearing in both E3 and DUB:", len(overlap_enz))
print(overlap_enz[:50])

overlap_df = pos[pos["enz_ac"].isin(overlap_enz)].sort_values(["enz_ac", "enzyme_class", "sub_ac"])
display(overlap_df.head(50))

overlap_df.to_csv(
    PROJECT_ROOT / "Data_proc/qc_reports/positive_e3_dub_accession_overlap.csv",
    index=False
)

چک خام ستون‌های UbiBrowser

In [ ]:
raw_files = [
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/E3-substrate-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/DUB-substrate-interactions.txt",
]

for f in raw_files:
    print("\n" + "="*100)
    print("FILE:", f.name)
    
    raw = pd.read_csv(f, sep=None, engine="python", dtype=str, nrows=5)
    
    print("shape:", raw.shape)
    print("columns:")
    print(raw.columns.tolist())
    display(raw)

چک فایل QC جدید

In [ ]:
qc_path = PROJECT_ROOT / "Data_proc/qc_reports/positive_qc_fixed.csv"

if qc_path.exists():
    qc = pd.read_csv(qc_path)
    display(qc)
else:
    print("positive_qc_fixed.csv not found.")

سلول اصلاح نهایی E3 و DUB

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(".")

e3_path = PROJECT_ROOT / "Data_proc/positives/positive_e3.csv"
dub_path = PROJECT_ROOT / "Data_proc/positives/positive_dub.csv"
all_path = PROJECT_ROOT / "Data_proc/positives/positive_all.csv"

e3 = pd.read_csv(e3_path)
dub = pd.read_csv(dub_path)

# Backup old separate files
e3.to_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3_before_class_fix.csv", index=False)
dub.to_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub_before_class_fix.csv", index=False)

# Force correct class
e3["enzyme_class"] = "E3"
dub["enzyme_class"] = "DUB"

# Rebuild IDs for each file
for df in [e3, dub]:
    df["enz_ac"] = df["enz_ac"].astype(str).str.strip()
    df["sub_ac"] = df["sub_ac"].astype(str).str.strip()
    
    df["pair_id"] = (
        df["enzyme_class"].astype(str)
        + "|"
        + df["enz_ac"].astype(str)
        + "|"
        + df["sub_ac"].astype(str)
    )
    
    df["group_id"] = (
        df["enzyme_class"].astype(str)
        + "|"
        + df["enz_ac"].astype(str)
    )

cols = [
    "pair_id",
    "group_id",
    "enzyme_class",
    "enz_ac",
    "sub_ac",
    "enz_gene",
    "sub_gene",
    "enzyme_type",
    "label",
    "source",
    "pmid",
]

e3 = e3[cols]
dub = dub[cols]

# Rebuild ALL again from fixed E3/DUB
positive_all = pd.concat([e3, dub], ignore_index=True)

# Save fixed files
e3.to_csv(e3_path, index=False)
dub.to_csv(dub_path, index=False)
positive_all.to_csv(all_path, index=False)

print("Saved fixed positive_e3.csv")
print("Saved fixed positive_dub.csv")
print("Saved fixed positive_all.csv")

بعدش دوباره چک نهایی را اجرا کن

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".")

e3 = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_e3.csv")
dub = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_dub.csv")
pos = pd.read_csv(PROJECT_ROOT / "Data_proc/positives/positive_all.csv")

for name, df in [("E3", e3), ("DUB", dub), ("ALL", pos)]:
    print("\n" + "="*80)
    print(name)
    print("shape:", df.shape)
    print("enzyme_class counts:")
    print(df["enzyme_class"].value_counts(dropna=False))
    print("label counts:")
    print(df["label"].value_counts(dropna=False))
    print("unique pair_id:", df["pair_id"].nunique())
    print("duplicate pair_id rows:", df.duplicated("pair_id").sum())
    print("missing enzyme_class:", df["enzyme_class"].isna().sum())
    print("missing enz_ac:", df["enz_ac"].isna().sum())
    print("missing sub_ac:", df["sub_ac"].isna().sum())
    print("pair_id starts with nan:", df["pair_id"].astype(str).str.startswith("nan|").sum())
    display(df.head(5))

QC نهایی را ذخیره

In [ ]:
def qc_row(df, name):
    return {
        "dataset": name,
        "n_rows": len(df),
        "n_unique_pair_id": df["pair_id"].nunique(),
        "n_duplicate_pair_id_rows": int(df.duplicated("pair_id").sum()),
        "n_unique_group_id": df["group_id"].nunique(),
        "n_unique_enz_ac": df["enz_ac"].nunique(),
        "n_unique_sub_ac": df["sub_ac"].nunique(),
        "n_missing_enzyme_class": int(df["enzyme_class"].isna().sum()),
        "n_missing_enz_ac": int(df["enz_ac"].isna().sum()),
        "n_missing_sub_ac": int(df["sub_ac"].isna().sum()),
        "n_missing_enz_gene": int(df["enz_gene"].isna().sum()),
        "n_missing_sub_gene": int(df["sub_gene"].isna().sum()),
        "n_missing_enzyme_type": int(df["enzyme_type"].isna().sum()),
        "n_missing_pmid": int(df["pmid"].isna().sum()),
        "n_pair_id_starts_with_nan": int(df["pair_id"].astype(str).str.startswith("nan|").sum()),
    }

qc = pd.DataFrame([
    qc_row(e3, "E3_positive_final"),
    qc_row(dub, "DUB_positive_final"),
    qc_row(pos, "ALL_positive_final"),
])

display(qc)

qc.to_csv(PROJECT_ROOT / "Data_proc/qc_reports/positive_qc_final.csv", index=False)